In [106]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Database Connection
engine = create_engine('postgresql://postgres:""@localhost:5432/profit_optimization')

def get_baseline_report():
    query = "SELECT * FROM v_master_profitability"
    df = pd.read_sql(query, engine)
    
    print("--- GLOBAL BASELINE PERFORMANCE ---")
    metrics = {
        "Total Gross Revenue": df['gross_revenue_usd'].sum(),
        "Total Net Revenue": df['net_revenue_usd'].sum(),
        "Total Net Profit": df['net_profit_usd'].sum(),
        "Overall Return Rate (%)": df['is_returned'].mean() * 100,
        "Average Profit Margin (%)": (df['net_profit_usd'].sum() / df['net_revenue_usd'].sum()) * 100
    }
    
    for k, v in metrics.items():
        print(f"{k}: {v:,.2f}")
    
    return df

df_master = get_baseline_report()

--- GLOBAL BASELINE PERFORMANCE ---
Total Gross Revenue: 1,980,857,202.94
Total Net Revenue: 1,802,268,864.68
Total Net Profit: -617,824,994.77
Overall Return Rate (%): 17.81
Average Profit Margin (%): -34.28


As of the beginning of 2024, the company's total gross revenue was $1.98 billion. Although net income was $1.80 billion, total net profit remained in negative territory at -$618 million due to high costs and returns. The average net profit margin was measured at -4, and the total return ratio was observed to be [percentage missing]. This picture reveals that our overall operational performance is critical for sustainability.

In [107]:
#At this stage, we determine which combination (Country x Channel x Segment) yields the most or the most losses.
def generate_profit_matrix(df):
    """Creates a matrix to identify profitable and problematic segments."""
    
    matrix = df.groupby(['country', 'channel', 'segment']).agg(
        order_count=('order_id', 'count'),
        avg_return_rate=('is_returned', 'mean'),
        total_profit_usd=('net_profit_usd', 'sum'),
        profit_margin_pct=('net_profit_usd', 'sum') # Temporary for calculation
    ).reset_index()
    
    # Calculate margin percentage correctly
    revenue_agg = df.groupby(['country', 'channel', 'segment'])['net_revenue_usd'].sum().reset_index()
    matrix = matrix.merge(revenue_agg, on=['country', 'channel', 'segment'])
    matrix['profit_margin_pct'] = (matrix['total_profit_usd'] / matrix['net_revenue_usd']) * 100
    matrix['profit_per_order'] = matrix['total_profit_usd'] / matrix['order_count']
    # Sort by most profitable
    matrix = matrix.sort_values(by='profit_margin_pct', ascending=False)
    
    print("\n--- PROFITABILITY MATRIX (Top 5 Segments) ---")
    print(matrix.head(5))
    
    return matrix

profit_matrix = generate_profit_matrix(df_master)


--- PROFITABILITY MATRIX (Top 5 Segments) ---
   country channel  segment  order_count  avg_return_rate  total_profit_usd  \
8       TR     B2B  Premium         7975         0.039624     -3.773039e+07   
11      TR     D2C  Premium        14769         0.041844     -2.521343e+06   
2       DE     B2B  Premium        12208         0.036943     -1.272855e+08   
7       TR     B2B      Mid        15700         0.139427     -3.453220e+07   
17     UAE     D2C  Premium        38905         0.039995     -4.102803e+06   

    profit_margin_pct  net_revenue_usd  profit_per_order  
8          -10.461873     3.606466e+08      -4731.083069  
11         -13.593655     1.854794e+07       -170.718606  
2          -22.752097     5.594452e+08     -10426.401512  
7          -28.499823     1.211663e+08      -2199.502883  
17         -30.689864     1.336859e+07       -105.456949  


In [108]:
import plotly.express as px
import plotly.graph_objects as go

def visualize_diagnostics(df, matrix):
    """Generates visual reports for baseline and matrix analysis."""
    
    # 1. PROFITABILITY HEATMAP (The Matrix)
    # This shows Country vs Segment and their margins
    fig_heatmap = px.density_heatmap(
        matrix, 
        x="segment", 
        y="country", 
        z="profit_margin_pct",
        facet_col="channel",
        color_continuous_scale="RdYlGn", # Red to Green
        title="Profit Margin % by Country, Segment, and Channel",
        labels={'profit_margin_pct': 'Margin (%)'}
    )
    fig_heatmap.show()

    # 2. PROFIT WATERFALL (The Money Leak)
    # This shows how Gross Revenue turns into Net Profit
    total_gross = df['gross_revenue_usd'].sum()
    total_shipping = df['total_shipping_cost'].sum()
    total_marketing = df['total_marketing_cost'].sum()
    # Returns loss: (Gross - Net)
    total_returns_loss = total_gross - df['net_revenue_usd'].sum()
    final_profit = df['net_profit_usd'].sum()

    fig_waterfall = go.Figure(go.Waterfall(
        name="Profitability Leak", orientation="v",
        measure=["relative", "relative", "relative", "relative", "total"],
        x=["Gross Revenue","COGS", "Returns Loss", "Shipping Costs", "Marketing Spend", "Labor Costs", "Net Profit"],
        textposition="outside",
        text=[f"${v/1e6:.1f}M" for v in [total_gross, -df['cogs'].sum(), -total_returns_loss, -total_shipping, -total_marketing, -df['labor_cost'].sum(), final_profit]],
        y=[total_gross, -df['cogs'].sum(), -total_returns_loss, -total_shipping, -total_marketing, -df['labor_cost'].sum(), final_profit],
        connector={"line": {"color": "rgb(63, 63, 63)"}},
    ))

    fig_waterfall.update_layout(title="Global Profit Waterfall (USD)", showlegend=False)
    fig_waterfall.show()

    # 3. RETURN RATES BY SEGMENT (The Risk Factor)
    fig_returns = px.bar(
        matrix, 
        x="segment", 
        y="avg_return_rate", 
        color="country",
        barmode="group",
        title="Average Return Rates by Segment and Country",
        labels={'avg_return_rate': 'Return Rate (0-1)'}
    )
    fig_returns.show()

    fig_heatmap_per_order = px.density_heatmap(
    matrix, x="segment", y="country", z="profit_per_order",
    facet_col="channel", color_continuous_scale="RdYlGn",
    title="Profit per Order by Segment, Country, Channel"
    )
    fig_heatmap_per_order.show()

# --- RUN VISUALS ---
visualize_diagnostics(df_master, profit_matrix)

The TR – B2B – Premium segment stands out as the company's most profitable segment. On the other hand, the Premium and Mid segments in DE and UAE, as well as the TR – D2C Premium segment, are incurring losses due to high costs and returns. Entry segments generally exhibit low or negative profitability. This analysis forms the basis for segment and channel-based prioritization.

--
High return rates directly impact net profit. When examined by segment, the return rate is 28% in the Entry segment, 14% in the Mid segment, and 4% in the Premium segment. This indicates that profits are rapidly eroding, particularly in low-margin sales within the Entry segment.

--
The TR – B2B Premium segment generates the highest profit per order, while the TR – D2C Premium segment shows low or negative profitability. The DE and UAE segments generally exhibit low profitability levels. This analysis is critical for understanding the operational efficiency of each segment and making strategic decisions.

--


In [109]:
# The "What-If" Simulator
import numpy as np

np.random.seed(42)

def run_simulation_engine(df, mkt_change=0.0, return_improve=0.0, shipping_cost_shock=0.0):
    """
    Simulates changes in marketing, returns, and shipping costs.
    Returns a dataframe with updated profits and prints impact.
    """

    sim_df = df.copy()

    # 1. Marketing değişimi
    sim_df['total_marketing_cost'] *= (1 + mkt_change)

    # 2. Segment bazlı return iyileştirme
    segment_impact = {
        'Entry': 0.3,
        'Mid': 0.2,
        'Premium': 0.1
    }

    for segment, impact in segment_impact.items():
        seg_mask = sim_df['segment'] == segment
        returned_idx = sim_df[seg_mask & (sim_df['is_returned'] == True)].index
        
        fix_count = int(len(returned_idx) * return_improve * impact)
        
        if fix_count > 0:
            fix_idx = np.random.choice(returned_idx, fix_count, replace=False)
            sim_df.loc[fix_idx, 'is_returned'] = False

    # 3. Net revenue yeniden hesapla
    sim_df['net_revenue_usd'] = np.where(
        sim_df['is_returned'],
        0,
        sim_df['gross_revenue_usd']
    )

    # 4. Shipping cost shock
    sim_df['total_shipping_cost'] *= (1 + shipping_cost_shock)

    # 5. YENİ COST MODEL
    sim_df['cogs'] = sim_df['base_price_usd'] * sim_df['quantity']
    sim_df['labor_cost'] = sim_df['gross_revenue_usd'] * sim_df['labor_ratio']

    # 6. FINAL PROFIT
    sim_df['new_net_profit'] = (
        sim_df['net_revenue_usd']
        - sim_df['cogs']
        - sim_df['total_shipping_cost']
        - sim_df['total_marketing_cost']
        - sim_df['labor_cost']
    )

    # Baseline & Simulation Impact
    old_profit = df['net_profit_usd'].sum()
    new_profit = sim_df['new_net_profit'].sum()
    profit_change = new_profit - old_profit

    print("\n--- SIMULATION RESULTS ---")
    print(f"Baseline Profit: ${old_profit:,.2f}")
    print(f"Simulated Profit: ${new_profit:,.2f}")

    # Doğru yorum: kar mı arttı, zarar mı büyüdü
    if old_profit < 0 and new_profit < 0:
        print(f"Loss Change: ${profit_change:,.2f} (loss {'decreased' if profit_change > 0 else 'increased'})")
    elif old_profit < 0 < new_profit:
        print(f"Profit Turnaround! From loss to profit: ${profit_change:,.2f}")
    else:
        delta_pct = (profit_change / abs(old_profit)) * 100
        print(f"Profit Growth: {delta_pct:+.2f}%")

    return sim_df

# Example Usage: Reduce returns by 20%, increase marketing 5%, shipping costs +10%
simulated_data = run_simulation_engine(
    df_master,
    mkt_change=0.05,
    return_improve=0.20,
    shipping_cost_shock=0.10
)


--- SIMULATION RESULTS ---
Baseline Profit: $-617,824,994.77
Simulated Profit: $-631,218,117.90
Loss Change: $-13,393,123.13 (loss increased)


In a scenario where returns are reduced by 20% and shipping costs increase by 10%, net profit is -$631 million, representing a $13.8 million greater loss than baseline. This indicates that short-term operational improvements need to be balanced against cost shocks.

In [110]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

def visualize_simulation_comparison(df_baseline, df_simulated, matrix_baseline, matrix_simulated):
    """
    CFO seviyesi karşılaştırma:
    1️⃣ Waterfall: Baseline vs Simulated
    2️⃣ Segment-wise Profit Shift
    3️⃣ Profit per Order Heatmap (Baseline ve Simulated)
    4️⃣ Net Margin Insight
    """

    # --- 1️⃣ Waterfall: Net Profit Breakdown ---
    waterfall_df = pd.DataFrame({
        'Category': ['Gross Revenue', 'COGS', 'Returns Loss', 'Shipping', 'Marketing', 'Labor', 'Net Profit'],
        'Baseline': [
            df_baseline['gross_revenue_usd'].sum(),
            df_baseline['cogs'].sum(),
            df_baseline['gross_revenue_usd'].sum() - df_baseline['net_revenue_usd'].sum(),
            df_baseline['total_shipping_cost'].sum(),
            df_baseline['total_marketing_cost'].sum(),
            df_baseline['labor_cost'].sum(),
            df_baseline['net_profit_usd'].sum()
        ],
        'Simulated': [
            df_simulated['gross_revenue_usd'].sum(),
            df_simulated['cogs'].sum(),
            df_simulated['gross_revenue_usd'].sum() - df_simulated['net_revenue_usd'].sum(),
            df_simulated['total_shipping_cost'].sum(),
            df_simulated['total_marketing_cost'].sum(),
            df_simulated['labor_cost'].sum(),
            df_simulated['new_net_profit'].sum()
        ]
    })

    # Waterfall için measure ve negatif değerleri ayarla
    def prep_waterfall_values(values):
        measure = ['relative'] * (len(values)-1) + ['total']
        y_values = values.copy()
        # maliyetleri negatif göster
        y_values[1:6] = [-v for v in y_values[1:6]]
        return measure, y_values

    fig_waterfall = go.Figure()
    for col, color_inc, color_dec, color_total in zip(
        ['Baseline','Simulated'],
        ['blue','green'],  # increasing
        ['red','orange'],  # decreasing
        ['darkblue','darkgreen']  # totals
    ):
        measure, y_values = prep_waterfall_values(waterfall_df[col].tolist())
        fig_waterfall.add_trace(go.Waterfall(
            name=col,
            x=waterfall_df['Category'],
            y=y_values,
            measure=measure,
            text=[f"${v/1e6:.1f}M" for v in y_values],
            textposition="outside",
            connector={"line":{"color":"rgb(63,63,63)"}},
            increasing={"marker":{"color":color_inc}},
            decreasing={"marker":{"color":color_dec}},
            totals={"marker":{"color":color_total}}
        ))

    fig_waterfall.update_layout(
        title="Global Profit Waterfall: Baseline vs Simulated",
        waterfallgap=0.5
    )
    fig_waterfall.show()

    # --- 2️⃣ Segment-wise Profit Shift ---
    seg_base = df_baseline.groupby('segment')['net_profit_usd'].sum().reset_index()
    seg_sim = df_simulated.groupby('segment')['new_net_profit'].sum().reset_index()
    seg_comp = seg_base.merge(seg_sim, on='segment')
    seg_comp.columns = ['Segment','Baseline','Simulated']
    seg_comp['Profit Change'] = seg_comp['Simulated'] - seg_comp['Baseline']

    fig_seg = px.bar(
        seg_comp,
        x='Segment',
        y='Profit Change',
        color='Profit Change',
        color_continuous_scale='RdYlGn',
        title="Segment-wise Profit Change: Simulated vs Baseline",
        labels={'Profit Change':'Δ Net Profit (USD)'}
    )
    fig_seg.show()

    # --- 3️⃣ Profit per Order Heatmap (Baseline vs Simulated) ---
    matrix_baseline['Scenario'] = 'Baseline'
    matrix_simulated['Scenario'] = 'Simulated'
    heatmap_df = pd.concat([matrix_baseline, matrix_simulated])

    fig_heatmap = px.density_heatmap(
        heatmap_df,
        x='segment',
        y='country',
        z='profit_per_order',
        facet_col='channel',
        color_continuous_scale='RdYlGn',
        facet_col_wrap=2,
        title="Profit per Order: Baseline vs Simulated",
        animation_frame='Scenario',
        labels={'profit_per_order':'Profit per Order (USD)'}
    )
    fig_heatmap.show()

    # --- 4️⃣ Margin Insight ---
    baseline_margin = df_baseline['net_profit_usd'].sum() / df_baseline['net_revenue_usd'].sum() * 100
    simulated_margin = df_simulated['new_net_profit'].sum() / df_simulated['net_revenue_usd'].sum() * 100
    print("\n--- 📈 MARGIN INSIGHT ---")
    print(f"Baseline Net Margin: %{baseline_margin:.2f}")
    print(f"Simulated Net Margin: %{simulated_margin:.2f}")
    print(f"Margin Improvement: {simulated_margin - baseline_margin:+.2f} percentage points")

# --- RUN VISUAL ---
profit_matrix_simulated = generate_profit_matrix(simulated_data)
visualize_simulation_comparison(df_master, simulated_data, profit_matrix, profit_matrix_simulated)


--- PROFITABILITY MATRIX (Top 5 Segments) ---
   country channel  segment  order_count  avg_return_rate  total_profit_usd  \
8       TR     B2B  Premium         7975         0.038746     -3.773039e+07   
11      TR     D2C  Premium        14769         0.041167     -2.521343e+06   
2       DE     B2B  Premium        12208         0.036206     -1.272855e+08   
7       TR     B2B      Mid        15700         0.133503     -3.453220e+07   
10      TR     D2C      Mid        29065         0.134320     -2.539117e+06   

    profit_margin_pct  net_revenue_usd  profit_per_order  
8          -10.453426     3.609380e+08      -4731.083069  
11         -13.586923     1.855713e+07       -170.718606  
2          -22.735213     5.598606e+08     -10426.401512  
7          -28.300927     1.220179e+08      -2199.502883  
10         -30.543452     8.313130e+06        -87.359948  



--- 📈 MARGIN INSIGHT ---
Baseline Net Margin: %-34.28
Simulated Net Margin: %-34.87
Margin Improvement: -0.59 percentage points


The simulation revealed varying impacts on different segments. While improved returns increased profits in some segments, increased shipping costs and other operational expenses resulted in net losses in others. This analysis provides guidance for measuring and prioritizing the impact of segment-specific actions.

--

While the baseline net margin was at -34%, the global net margin declined to -35% after the simulation. Although segment-based improvements resulted in some profit increases, the global impact was negative. This result highlights the need for more proactive action in cost and return management.

In [111]:
def generate_profit_matrix(df):

    matrix = df.groupby(['country', 'channel', 'segment']).agg(
        order_count=('order_id', 'count'),
        avg_return_rate=('is_returned', 'mean'),
        total_profit_usd=('net_profit_usd', 'sum')
    ).reset_index()

    revenue_agg = df.groupby(['country', 'channel', 'segment'])['net_revenue_usd'].sum().reset_index()
    matrix = matrix.merge(revenue_agg, on=['country', 'channel', 'segment'])

    matrix['profit_margin_pct'] = (matrix['total_profit_usd'] / matrix['net_revenue_usd']) * 100

    # 🔥 NEW
    matrix['profit_per_order'] = matrix['total_profit_usd'] / matrix['order_count']

    return matrix.sort_values(by='profit_margin_pct', ascending=False)

import plotly.express as px

def visualize_profit_matrix(matrix):
    """
    Segment bazlı kâr ve kâr marjını görselleştirir.
    """
    # 1️⃣ Profit Margin Heatmap
    fig_margin = px.density_heatmap(
        matrix,
        x='segment',
        y='country',
        z='profit_margin_pct',
        facet_col='channel',
        color_continuous_scale='RdYlGn',  # Kırmızı = düşük, Yeşil = yüksek
        title="Profit Margin % by Country, Segment, Channel",
        labels={'profit_margin_pct':'Profit Margin (%)'}
    )
    fig_margin.show()
    
    # 2️⃣ Profit per Order Heatmap
    fig_per_order = px.density_heatmap(
        matrix,
        x='segment',
        y='country',
        z='profit_per_order',
        facet_col='channel',
        color_continuous_scale='RdYlGn',
        title="Profit per Order by Country, Segment, Channel",
        labels={'profit_per_order':'Profit per Order (USD)'}
    )
    fig_per_order.show()

# --- RUN VISUAL ---
visualize_profit_matrix(profit_matrix)

In [112]:
import numpy as np
import pandas as pd

def simulate_scenario(df, mkt_change=0.0, return_improve=0.0, shipping_cost_shock=0.0):
    """
    Decision Engine: What-If Simulator
    - mkt_change: marketing spend change (0.2 = +20%)
    - return_improve: overall return reduction factor (0.2 = -20%)
    - shipping_cost_shock: shipping cost increase (0.1 = +10%)
    
    Returns:
        dict: JSON-ready summary with new margin, profit change, best segment
    """
    
    sim_df = df.copy()
    
    # 1️⃣ Marketing change
    sim_df['total_marketing_cost'] *= (1 + mkt_change)
    
    # 2️⃣ Return improvement by segment
    segment_impact = {'Entry': 0.3, 'Mid': 0.2, 'Premium': 0.1}
    for segment, impact in segment_impact.items():
        seg_mask = sim_df['segment'] == segment
        returned_idx = sim_df[seg_mask & sim_df['is_returned']].index
        fix_count = int(len(returned_idx) * return_improve * impact)
        if fix_count > 0:
            fix_idx = np.random.choice(returned_idx, fix_count, replace=False)
            sim_df.loc[fix_idx, 'is_returned'] = False
    
    # 3️⃣ Net revenue recalculation
    sim_df['net_revenue_usd'] = np.where(sim_df['is_returned'], 0, sim_df['gross_revenue_usd'])
    
    # 4️⃣ Shipping cost shock
    sim_df['total_shipping_cost'] *= (1 + shipping_cost_shock)
    
    # 5️⃣ Recalculate costs
    sim_df['cogs'] = sim_df['base_price_usd'] * sim_df['quantity']
    sim_df['labor_cost'] = sim_df['gross_revenue_usd'] * sim_df['labor_ratio']
    
    # 6️⃣ Final profit
    sim_df['new_net_profit'] = sim_df['net_revenue_usd'] - sim_df['cogs'] - sim_df['total_shipping_cost'] - sim_df['total_marketing_cost'] - sim_df['labor_cost']
    
    # 7️⃣ KPI calculations
    baseline_profit = df['net_profit_usd'].sum()
    new_profit = sim_df['new_net_profit'].sum()
    profit_change_pct = ((new_profit - baseline_profit) / abs(baseline_profit)) * 100
    
    # 8️⃣ Identify Best Segment (country x channel x segment with max profit)
    segment_summary = sim_df.groupby(['country','channel','segment'])['new_net_profit'].sum().reset_index()
    best_row = segment_summary.loc[segment_summary['new_net_profit'].idxmax()]
    best_segment = f"{best_row['country']} {best_row['channel']} {best_row['segment']}"
    
    # 9️⃣ Net margin
    new_margin = new_profit / sim_df['net_revenue_usd'].sum() if sim_df['net_revenue_usd'].sum() != 0 else 0
    
    #  🔟 JSON-ready output
    result = {
        "new_margin": round(new_margin, 4),
        "profit_change": f"{profit_change_pct:+.2f}%",
        "best_segment": best_segment
    }
    
    return result, sim_df

# --- Örnek Kullanım ---
result_json, simulated_df = simulate_scenario(
    df_master,
    mkt_change=0.05,        # Marketing +5%
    return_improve=0.20,    # Return reduction 20%
    shipping_cost_shock=0.10 # Shipping cost +10%
)

print(result_json)

{'new_margin': -0.3489, 'profit_change': '-2.21%', 'best_segment': 'UAE D2C Entry'}
